# Phase 2 - Data Cleaning Pipeline

Every cleaning step is a **documented function**, not a throwaway one-liner. After
each step we log how much data is affected (**data-loss tracking**). The output is
two clean files that everything downstream (SQL, Power BI, stats) reads from.

A pipeline = *reproducible* transformations: raw in -> documented steps -> clean out.
Run it again tomorrow on new data and it behaves identically.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/india_all_restaurants_details.csv", low_memory=False)
print("Loaded raw:", df.shape)

Loaded raw: (224520, 18)


### Step 1 - Clean the `rating` column
`'4.1'` -> `4.1`; `'NEW'`, `'0'`, garbage -> `NaN`. We treat `NEW`/`0` as *no rating*
(a new restaurant) rather than a real zero, which would wrongly drag averages down.

In [2]:
NON_RATINGS = {"new", "0", "-", "", "nan", "none"}

def clean_rating(value):
    """'4.1' -> 4.1 ; 'NEW'/'0'/garbage/NaN -> NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text.lower() in NON_RATINGS:
        return np.nan
    try:
        return float(text)
    except ValueError:
        return np.nan

df["rating"] = df["rating"].apply(clean_rating)
print(f"Ratings now numeric. Unrated (NaN): {df['rating'].isna().sum():,} rows "
      f"({100*df['rating'].isna().mean():.1f}%)")

Ratings now numeric. Unrated (NaN): 79,785 rows (35.5%)


### Step 2 - Clean the `cost_for_two` column
`'1,200'` -> `1200.0`; commas are thousands separators. Cost `0` is invalid -> `NaN`.

In [3]:
def clean_cost(value):
    """'1,200' -> 1200.0 ; ''/0/NaN -> NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value).replace(",", "").strip()
    try:
        cost = float(text)
    except ValueError:
        return np.nan
    return cost if cost > 0 else np.nan

df["cost_for_two"] = df["cost_for_two"].apply(clean_cost)
print(f"Cost cleaned. Invalid/missing: {df['cost_for_two'].isna().sum():,} rows")
print(df["cost_for_two"].describe())

Cost cleaned. Invalid/missing: 3,648 rows
count    220872.000000
mean        425.365904
std         375.092960
min           2.000000
25%         200.000000
50%         300.000000
75%         500.000000
max       30000.000000
Name: cost_for_two, dtype: float64


### Step 3 - Standardise city names
The same real-world city must always be one label. We fold known aliases
(e.g. Gurgaon/Noida/New Delhi -> `Delhi NCR`) so counts don't split.

In [4]:
CITY_MAP = {
    "Bangalore": "Bengaluru", "Mysore": "Mysuru", "Pondicherry": "Puducherry",
    "Gurgaon": "Delhi NCR", "Noida": "Delhi NCR", "New Delhi": "Delhi NCR", "Delhi": "Delhi NCR",
}

def standardise_city(value):
    if pd.isna(value):
        return np.nan
    city = str(value).strip()
    return CITY_MAP.get(city, city)

df["city_clean"] = df["city"].apply(standardise_city)
print("Distinct cities after standardising:", df["city_clean"].nunique())

Distinct cities after standardising: 83


### Step 4 - Booleans, votes, location + engineered features
The Yes/No fields are already boolean here. We rename for clarity, treat
`rating_count` as `votes`, and engineer `price_tier` and `city_tier`.

In [5]:
df["has_online_order"] = df["online_order"].astype(bool)
df["has_table_booking"] = df["table_reservation"].astype(bool)
df["votes"] = pd.to_numeric(df["rating_count"], errors="coerce").fillna(0).astype(int)
df["location"] = df["area"].astype(str).str.strip()

def price_tier(cost):
    if pd.isna(cost):      return "Unknown"
    if cost < 300:         return "Budget (<300)"
    if cost < 700:         return "Mid-range (300-700)"
    if cost < 1500:        return "Premium (700-1500)"
    return "Luxury (1500+)"

TIER1 = {"Delhi NCR", "Mumbai", "Bengaluru", "Hyderabad",
         "Chennai", "Pune", "Kolkata", "Ahmedabad"}

df["price_tier"] = df["cost_for_two"].apply(price_tier)
df["city_tier"] = df["city_clean"].apply(lambda c: "Tier 1" if c in TIER1 else "Tier 2+")

print(df["price_tier"].value_counts())
print()
print(df["city_tier"].value_counts())

price_tier
Mid-range (300-700)    115881
Budget (<300)           76739
Premium (700-1500)      22661
Luxury (1500+)           5591
Unknown                  3648
Name: count, dtype: int64

city_tier
Tier 1     140417
Tier 2+     84103
Name: count, dtype: int64


### Step 5 - Explode the `cusine` column (normalisation)
One cell can hold up to 8 cuisines. To count cuisines correctly we create one
row per (restaurant, cuisine). Use this table **only** for cuisine-level analysis,
or you'd multi-count restaurants.

In [6]:
keep = ["name", "city_clean", "city_tier", "location", "cusine",
        "rating", "votes", "cost_for_two", "price_tier",
        "has_online_order", "has_table_booking", "delivery_only"]
clean = df[keep].copy()

tmp = clean.copy()
tmp["cusine"] = tmp["cusine"].fillna("Unknown")
cuisines = tmp.assign(cuisine=tmp["cusine"].str.split(",")).explode("cuisine")
cuisines["cuisine"] = cuisines["cuisine"].str.strip()
cuisines = cuisines[cuisines["cuisine"] != ""][["name", "city_clean", "cuisine", "rating", "cost_for_two"]]

print("Restaurants:", len(clean), "| Cuisine rows after explode:", len(cuisines))
print(cuisines["cuisine"].value_counts().head(10))

Restaurants: 224520 | Cuisine rows after explode: 473264
cuisine
North Indian    88565
Chinese         62685
Fast Food       56872
South Indian    26111
Beverages       22401
Desserts        21997
Biryani         17171
Bakery          16749
Street Food     14193
Mughlai         11472
Name: count, dtype: int64


### Step 6 - Validate & export
Quick before/after sanity check, then write the two clean files. These are the
exact same outputs `pipeline/run_pipeline.py` produces - the notebook is the
"explain it" version, the pipeline is the "automate it" version.

In [7]:
import os
structurally_valid = clean["cost_for_two"].notna() & clean["city_clean"].notna()
print("Rows read     :", len(df))
print("Rows kept     :", len(clean))
print("Valid cost+city: {:,} ({:.2f}%)".format(
      structurally_valid.sum(), 100*structurally_valid.mean()))

os.makedirs("../data/cleaned", exist_ok=True)
clean.to_csv("../data/cleaned/zomato_clean.csv", index=False)
cuisines.to_csv("../data/cleaned/zomato_cuisines.csv", index=False)
print("\nExported zomato_clean.csv and zomato_cuisines.csv")

Rows read     : 224520
Rows kept     : 224520
Valid cost+city: 220,872 (98.38%)



Exported zomato_clean.csv and zomato_cuisines.csv
